In [1]:
!pip install langgraph

In [3]:
from langgraph.graph import StateGraph
from langchain_groq import ChatGroq

In [4]:
api_key = "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"

In [5]:
llm = ChatGroq(
    model = "llama-3.3-70b-versatile",
    groq_api_key = api_key,
    temperature = 0.5,
    max_tokens = 128
)

In [7]:
from typing import TypedDict , Optional

class AuthState(TypedDict):
    username : Optional[str]
    password : Optional[str]
    is_authenticated: Optional[bool]
    output: Optional[str]

In [8]:
#Success login example
auth_state_1: AuthState = {
    "username": "alice123",
    "password": "123",
    "is_authenticated": True,
    "output": "Login successful."
}
print(f"auth_state_1: {auth_state_1}")

auth_state_1: {'username': 'alice123', 'password': '123', 'is_authenticated': True, 'output': 'Login successful.'}


In [9]:
#Fail login example
auth_state_2: AuthState = {
    "username":"",
    "password": "wrongpassword",
    "is_authenticated": False,
    "output": "Authentication failed. Please try again."
}
print(f"auth_state_2: {auth_state_2}")

auth_state_2: {'username': '', 'password': 'wrongpassword', 'is_authenticated': False, 'output': 'Authentication failed. Please try again.'}


In [10]:
def input_node(state):
    print(state)
    if state.get("username" ,"")== "":
        state["username"] = input("What is your username")
    password = input("What is your password?.")
    return {"password":password}

In [11]:
input_node(auth_state_1)

{'username': 'alice123', 'password': '123', 'is_authenticated': True, 'output': 'Login successful.'}


What is your password?. 123


{'password': '123'}

In [12]:
input_node(auth_state_2)

{'username': '', 'password': 'wrongpassword', 'is_authenticated': False, 'output': 'Authentication failed. Please try again.'}


What is your username alice
What is your password?. 123


{'password': '123'}

In [49]:
# Validation node
def validate_credentials_node(state):
    username = state.get("username","")
    password = state.get("password", "")
    print("Username ",username, "Password", password)
    #Simulated credential validation
    if username == "test_user" and password == "secure_password":
        is_authenticated = True
    else:
        is_authenticated = False
    return {"is_authenticated":is_authenticated}    

In [51]:
validate_credentials_node(auth_state_1)
auth_state_1

Username  alice123 Password 123


{'username': 'alice123',
 'password': '123',
 'is_authenticated': True,
 'output': 'Login successful.'}

In [15]:
auth_state_3: AuthState = {
    "username":"test_user",
    "password":  "secure_password",
    "is_authenticated": False,
    "output": "Authentication failed. Please try again."
}
print(f"auth_state_3: {auth_state_3}")

auth_state_3: {'username': 'test_user', 'password': 'secure_password', 'is_authenticated': False, 'output': 'Authentication failed. Please try again.'}


In [16]:
validate_credentials_node(auth_state_3)

Username  test_user Password secure_password


{'is_authenticated': True}

In [45]:
#Define the success node
def success_node(state):
    return {"output": "Authentication Successful!.Welcome."}

In [46]:
success_node(auth_state_3)
auth_state_3

{'username': 'test_user',
 'password': 'secure_password',
 'is_authenticated': False,
 'output': 'Authentication failed. Please try again.'}

In [20]:
# Define the failure node
def failure_node(state):
    return {"output": "Not Successfull, please try again!"}

In [47]:
failure_node(auth_state_2)
auth_state_2

{'username': 'alice',
 'password': 'wrongpassword',
 'is_authenticated': False,
 'output': 'Authentication failed. Please try again.'}

In [22]:
#Router node
def router(state):
    if state["is_authenticated"]:
        return "success_node"
    else:
        return "failure_node"

In [23]:
from langgraph.graph import StateGraph , END

In [24]:
workflow = StateGraph(AuthState)
workflow

In [25]:
workflow.add_node("InputNode",input_node)


In [26]:
workflow.add_node("ValidateNode", validate_credentials_node)
workflow.add_node("Success", success_node)
workflow.add_node("Failure", failure_node)

In [27]:
workflow.add_edge("InputNode" , "ValidateNode")


In [28]:
workflow.add_edge("Success", END)
workflow.add_edge("Failure", END)

In [29]:
#Add conditional edges
workflow.add_conditional_edges("ValidateNode",router,{"success_node":"Success","failure_node":"Failure"})

In [30]:
#Setting the starting or entry point to the graph
workflow.set_entry_point("InputNode")

In [31]:
#Compiling the workflow or graph
app = workflow.compile()

In [32]:
#Running the application
inputs = {"username": "test_user"}
result = app.invoke(inputs)
print(result)

{'username': 'test_user'}


What is your password?. secure_password


Username  test_user Password secure_password
{'username': 'test_user', 'password': 'secure_password', 'is_authenticated': True, 'output': 'Authentication Successful!.Welcome.'}


In [57]:
result["output"]
print(app.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	InputNode(InputNode)
	ValidateNode(ValidateNode)
	Success(Success)
	Failure(Failure)
	__end__([<p>__end__</p>]):::last
	InputNode --> ValidateNode;
	ValidateNode -. &nbsp;failure_node&nbsp; .-> Failure;
	ValidateNode -. &nbsp;success_node&nbsp; .-> Success;
	__start__ --> InputNode;
	Failure --> __end__;
	Success --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [34]:
#Define structure of QA state
class QAState(TypedDict):
     # 'question' stores the user's input question. It can be a string or None if not provided.
    question: Optional[str]
    
    # 'context' stores relevant context about the guided project, if the question pertains to it.
    # If the question isn't related to the project, this will be None.
    context: Optional[str]
    
    # 'answer' stores the generated response or answer. It can be None until the answer is generated.
    answer: Optional[str]

In [35]:
# Create an example object
qa_state_example = QAState(
    question="What is the purpose of this guided project?",
    context="This project focuses on building a chatbot using Python.",
    answer=None
)

# Print the attributes
for key, value in qa_state_example.items():
    print(f"{key}: {value}")

question: What is the purpose of this guided project?
context: This project focuses on building a chatbot using Python.
answer: None


In [36]:
def input_validation_node(state):
    # Extract the question from the state, and strip any leading or trailing spaces
    question = state.get("question", "").strip()
    #If question is empty return an error message
    if not question:
        return {"valid":False,"Error": "Invalid input.Question must not be empty."}
    return {"valid": True}    

In [37]:
input_validation_node(qa_state_example)

{'valid': True}

In [38]:
def context_provider_node(state):
    question = state.get("question", "").lower()
    # Check if the question is related to the guided project
    if "langgraph" in question or "guided project" in question:
        context = (
            "This guided project is about using LangGraph, a Python library to design state-based workflows. "
            "LangGraph simplifies building complex applications by connecting modular nodes with conditional edges."
        )
        return {"context": context}
    # If unrelated, set context to null
    return {"context": None}

In [39]:
def llm_qa_node(state):
    #Extract the question and context from state
    question = state.get("question","")
    context = state.get("context" , None)
    if not context:
        return {"answer" : "I don't have enough context to answer your question"}
    prompt = f"Context: {context}\nQuestion: {question}\nAnswer the question based on the provided context."
    try:
        response = llm.invoke(prompt)
        return {"answer": response.content.strip()}
    except Exception as e:
        return {"answer": f"An error ocuured :{str(e)}"}
    

In [40]:
qa_workflow = StateGraph(QAState)

In [41]:
qa_workflow.add_node("InputNode" , input_validation_node)
qa_workflow.add_node("ContextNode", context_provider_node)
qa_workflow.add_node("QANode" , llm_qa_node)

In [42]:
qa_workflow.set_entry_point("InputNode")

In [43]:
qa_workflow.add_edge("InputNode" , "ContextNode")


In [52]:
qa_workflow.add_edge("ContextNode" , "QANode")
qa_workflow.add_edge("QANode", END)

In [53]:
qa_app = qa_workflow.compile()

In [54]:
qa_app.invoke({"question": "What is the weather today?"})

{'question': 'What is the weather today?',
 'context': None,
 'answer': "I don't have enough context to answer your question"}

In [56]:
qa_app.invoke({"question": "What is LangGraph?"})

{'question': 'What is LangGraph?',
 'context': 'This guided project is about using LangGraph, a Python library to design state-based workflows. LangGraph simplifies building complex applications by connecting modular nodes with conditional edges.',
 'answer': 'LangGraph is a Python library used to design state-based workflows by connecting modular nodes with conditional edges, simplifying the process of building complex applications.'}